# 08 — Synthèse des benchmarks et présélection des candidats

Ce notebook est prêt à l’emploi pour ton projet. Il lit les résultats produits par `NB1` à `NB4`, les affiche **un à un** sans concaténation, les classe à l’intérieur de chaque notebook, puis retient les **2 meilleurs pipelines** de chaque groupe.

Cette version a été ajustée pour tenir compte des **vrais noms de fichiers** exportés dans tes dossiers `outputs/NB1`, `outputs/NB2`, `outputs/NB3` et `outputs/NB4`.


## Installation éventuelle

Décommente cette cellule si nécessaire.


In [16]:
# %pip install -q pandas openpyxl

In [17]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)


## Paramètres

Ce notebook est prévu pour être exécuté depuis `notebooks/models_training/`.

Les chemins utilisés ci-dessous sont alignés avec ton architecture actuelle :
- notebooks dans `notebooks/models_training/`
- résultats dans `outputs/NB1`, `outputs/NB2`, `outputs/NB3`, `outputs/NB4`

Le notebook cherche plusieurs noms possibles pour chaque fichier, en commençant par les noms **réellement observés** dans tes exports.


In [18]:
OUTPUTS_ROOT = Path("../../outputs")

RESULT_FILES = {
    "NB1": [
        OUTPUTS_ROOT / "NB1" / "NB1_count_tfidf_baselines_resultats_selection.csv",
        OUTPUTS_ROOT / "NB1" / "NB1_count_tfidf_baselines_resultats_par_pipeline.csv",
        OUTPUTS_ROOT / "NB1" / "NB1_resultats_selection.csv",
        OUTPUTS_ROOT / "NB1" / "NB1_resultats_par_pipeline.csv",
    ],
    "NB2": [
        OUTPUTS_ROOT / "NB2" / "NB2_weighting_char_hybrid_resultats_selection.csv",
        OUTPUTS_ROOT / "NB2" / "NB2_weighting_char_hybrid_resultats_par_pipeline.csv",
        OUTPUTS_ROOT / "NB2" / "NB2_resultats_selection.csv",
        OUTPUTS_ROOT / "NB2" / "NB2_resultats_par_pipeline.csv",
    ],
    "NB3": [
        OUTPUTS_ROOT / "NB3" / "NB3_reduction_selection_nb_resultats_selection.csv",
        OUTPUTS_ROOT / "NB3" / "NB3_reduction_selection_nb_resultats_par_pipeline.csv",
        OUTPUTS_ROOT / "NB3" / "NB3_resultats_selection.csv",
        OUTPUTS_ROOT / "NB3" / "NB3_resultats_par_pipeline.csv",
    ],
}

TOP_K = {
    "NB1": 2,
    "NB2": 2,
    "NB3": 2,
}

PRIORITY_METRICS = [
    "test_f1_class_1",
    "test_recall_class_1",
    "test_f1_macro",
    "test_balanced_accuracy",
    "test_roc_auc",
]


## Fonctions utilitaires

Les fonctions ci-dessous servent à charger les fichiers de résultats, construire une table de revue avec les colonnes les plus utiles, puis classer les pipelines à l’intérieur de chaque notebook.


In [19]:
def load_first_existing(paths, notebook_name):
    for path in paths:
        if Path(path).exists():
            df = pd.read_csv(path)
            print(f"{notebook_name} -> fichier chargé : {path}")
            return df, Path(path)
    raise FileNotFoundError(
        f"Aucun fichier trouvé pour {notebook_name}. "
        f"Chemins testés : {[str(p) for p in paths]}"
    )


def prepare_review_table(df: pd.DataFrame) -> pd.DataFrame:
    review_cols = [
        "pipeline",
        "train_f1_class_1", "test_f1_class_1",
        "train_recall_class_1", "test_recall_class_1",
        "train_precision_class_1", "test_precision_class_1",
        "train_f1_macro", "test_f1_macro",
        "train_balanced_accuracy", "test_balanced_accuracy",
        "train_roc_auc", "test_roc_auc",
        "train_pr_auc", "test_pr_auc",
    ]
    existing = [c for c in review_cols if c in df.columns]
    out = df[existing].copy()

    if {"train_f1_class_1", "test_f1_class_1"}.issubset(out.columns):
        out["gap_f1_class_1"] = (out["train_f1_class_1"] - out["test_f1_class_1"]).abs()

    if {"train_f1_macro", "test_f1_macro"}.issubset(out.columns):
        out["gap_f1_macro"] = (out["train_f1_macro"] - out["test_f1_macro"]).abs()

    return out


def rank_pipelines(df: pd.DataFrame) -> pd.DataFrame:
    ranked = prepare_review_table(df).copy()

    sort_cols = []
    ascending = []

    for col in PRIORITY_METRICS:
        if col in ranked.columns:
            sort_cols.append(col)
            ascending.append(False)

    for gap_col in ["gap_f1_class_1", "gap_f1_macro"]:
        if gap_col in ranked.columns:
            sort_cols.append(gap_col)
            ascending.append(True)

    if not sort_cols:
        raise ValueError(
            "Aucune colonne de tri pertinente n'a été trouvée dans le DataFrame."
        )

    ranked = ranked.sort_values(by=sort_cols, ascending=ascending).reset_index(drop=True)
    ranked.insert(0, "rang", range(1, len(ranked) + 1))
    return ranked


## Chargement et affichage des résultats bruts

Aucune concaténation n’est réalisée. Chaque tableau est lu et affiché séparément.


In [20]:
raw_results = {}
loaded_paths = {}

for notebook_name, candidate_paths in RESULT_FILES.items():
    print("=" * 100)
    print(f"Résultats bruts -> {notebook_name}")
    df, used_path = load_first_existing(candidate_paths, notebook_name)
    raw_results[notebook_name] = df.copy()
    loaded_paths[notebook_name] = used_path
    display(df)


Résultats bruts -> NB1
NB1 -> fichier chargé : ..\..\outputs\NB1\NB1_count_tfidf_baselines_resultats_par_pipeline.csv


,pipeline,train_accuracy,train_precision_macro,train_recall_macro,train_f1_macro,train_precision_weighted,train_recall_weighted,train_f1_weighted,train_precision_class_0,train_recall_class_0,train_f1_class_0,train_support_class_0,train_precision_class_1,train_recall_class_1,train_f1_class_1,train_support_class_1,train_balanced_accuracy,train_roc_auc,train_pr_auc,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_precision_weighted,test_recall_weighted,test_f1_weighted,test_precision_class_0,test_recall_class_0,test_f1_class_0,test_support_class_0,test_precision_class_1,test_recall_class_1,test_f1_class_1,test_support_class_1,test_balanced_accuracy,test_roc_auc,test_pr_auc
0,P01_Count_MultinomialNB,0.9291,0.8767,0.8958,0.8858,0.9312,0.9291,0.9300,0.9635,0.9488,0.9561,7405.0,0.7899,0.8427,0.8155,1691.0,0.8958,0.9610,0.8981,0.8799,0.7991,0.8177,0.8078,0.8841,0.8799,0.8818,0.9345,0.9168,0.9256,1851.0,0.6638,0.7187,0.6901,423.0,0.8177,0.9054,0.7616
1,P02_TFIDF_Unigram_LogReg,0.9037,0.9319,0.7476,0.8011,0.9100,0.9037,0.8909,0.8970,0.9961,0.9439,7405.0,0.9668,0.4991,0.6583,1691.0,0.7476,0.9638,0.8970,0.8738,0.8601,0.6890,0.7324,0.8705,0.8738,0.8545,0.8767,0.9833,0.9269,1851.0,0.8434,0.3948,0.5378,423.0,0.6890,0.9133,0.7502
2,P03_TFIDF_UniBi_LogReg,0.8986,0.9376,0.7301,0.7851,0.9080,0.8986,0.8832,0.8904,0.9984,0.9413,7405.0,0.9849,0.4619,0.6288,1691.0,0.7301,0.9768,0.9313,0.8698,0.8789,0.6665,0.7101,0.8723,0.8698,0.8452,0.8683,0.9903,0.9253,1851.0,0.8896,0.3428,0.4949,423.0,0.6665,0.9157,0.7677
3,P04_TFIDF_UniBi_LinearSVC,0.9956,0.9957,0.9898,0.9927,0.9956,0.9956,0.9956,0.9956,0.9991,0.9973,7405.0,0.9958,0.9805,0.9881,1691.0,0.9898,0.9999,0.9993,0.8958,0.8472,0.7873,0.8122,0.8904,0.8958,0.8909,0.9160,0.9600,0.9375,1851.0,0.7784,0.6147,0.6869,423.0,0.7873,0.9161,0.7834
4,P05_TFIDF_UniBi_SGDLog,0.9963,0.9959,0.9918,0.9938,0.9963,0.9963,0.9963,0.9965,0.9989,0.9977,7405.0,0.9952,0.9846,0.9899,1691.0,0.9918,0.9999,0.9994,0.8997,0.8589,0.7889,0.8172,0.8947,0.8997,0.8943,0.9159,0.9654,0.9400,1851.0,0.8019,0.6123,0.6944,423.0,0.7889,0.9203,0.7843


Résultats bruts -> NB2
NB2 -> fichier chargé : ..\..\outputs\NB2\NB2_weighting_char_hybrid_resultats_par_pipeline.csv


,pipeline,train_accuracy,train_precision_macro,train_recall_macro,train_f1_macro,train_precision_weighted,train_recall_weighted,train_f1_weighted,train_precision_class_0,train_recall_class_0,train_f1_class_0,train_support_class_0,train_precision_class_1,train_recall_class_1,train_f1_class_1,train_support_class_1,train_balanced_accuracy,train_roc_auc,train_pr_auc,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_precision_weighted,test_recall_weighted,test_f1_weighted,test_precision_class_0,test_recall_class_0,test_f1_class_0,test_support_class_0,test_precision_class_1,test_recall_class_1,test_f1_class_1,test_support_class_1,test_balanced_accuracy,test_roc_auc,test_pr_auc
0,P06_TFIDF_Sublinear_LogReg,0.8989,0.9372,0.7309,0.7858,0.9080,0.8989,0.8836,0.8907,0.9982,0.9414,7405.0,0.9837,0.4636,0.6302,1691.0,0.7309,0.9773,0.9324,0.8707,0.8822,0.6680,0.7120,0.8738,0.8707,0.8463,0.8688,0.9908,0.9258,1851.0,0.8957,0.3452,0.4983,423.0,0.6680,0.9165,0.7687
1,P07_CharTFIDF_LogReg,0.9070,0.9341,0.7565,0.8098,0.9128,0.9070,0.8952,0.9003,0.9961,0.9458,7405.0,0.9679,0.5169,0.6739,1691.0,0.7565,0.9709,0.9111,0.8703,0.8567,0.6787,0.7211,0.8669,0.8703,0.8492,0.8730,0.9838,0.9251,1851.0,0.8404,0.3735,0.5172,423.0,0.6787,0.9087,0.7455
2,P08_CharTFIDF_LinearSVC,0.9919,0.9929,0.9802,0.9864,0.9919,0.9919,0.9918,0.9913,0.9988,0.9950,7405.0,0.9945,0.9616,0.9778,1691.0,0.9802,0.9997,0.9986,0.8835,0.8242,0.7643,0.7887,0.8765,0.8835,0.8776,0.9075,0.9541,0.9302,1851.0,0.7409,0.5745,0.6471,423.0,0.7643,0.9141,0.7524
3,P09_WordChar_Hybrid_LogReg,0.9394,0.9594,0.8407,0.8858,0.9425,0.9394,0.9350,0.9325,0.9978,0.9641,7405.0,0.9863,0.6836,0.8075,1691.0,0.8407,0.9899,0.9658,0.8839,0.8637,0.7226,0.7655,0.8797,0.8839,0.8701,0.8892,0.9795,0.9321,1851.0,0.8383,0.4657,0.5988,423.0,0.7226,0.9233,0.7809
4,P10_WordChar_Hybrid_LinearSVC,0.9989,0.9986,0.9977,0.9982,0.9989,0.9989,0.9989,0.9991,0.9996,0.9993,7405.0,0.9982,0.9959,0.9970,1691.0,0.9977,1.0000,1.0000,0.8918,0.8363,0.7858,0.8073,0.8863,0.8918,0.8874,0.9160,0.9546,0.9349,1851.0,0.7565,0.6170,0.6797,423.0,0.7858,0.9200,0.7757


Résultats bruts -> NB3
NB3 -> fichier chargé : ..\..\outputs\NB3\NB3_reduction_selection_nb_resultats_par_pipeline.csv


,pipeline,train_accuracy,train_precision_macro,train_recall_macro,train_f1_macro,train_precision_weighted,train_recall_weighted,train_f1_weighted,train_precision_class_0,train_recall_class_0,train_f1_class_0,train_support_class_0,train_precision_class_1,train_recall_class_1,train_f1_class_1,train_support_class_1,train_balanced_accuracy,train_roc_auc,train_pr_auc,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_precision_weighted,test_recall_weighted,test_f1_weighted,test_precision_class_0,test_recall_class_0,test_f1_class_0,test_support_class_0,test_precision_class_1,test_recall_class_1,test_f1_class_1,test_support_class_1,test_balanced_accuracy,test_roc_auc,test_pr_auc
0,P11_TFIDF_SVD_LogReg,0.8830,0.8595,0.7224,0.7644,0.8782,0.8830,0.8694,0.8893,0.9781,0.9316,7405.0,0.8297,0.4666,0.5973,1691.0,0.7224,0.8972,0.7405,0.8690,0.8085,0.7152,0.7472,0.8585,0.8690,0.8574,0.8881,0.9600,0.9226,1851.0,0.7289,0.4704,0.5718,423.0,0.7152,0.8852,0.6726
1,P12_TFIDF_SVD_LinearSVC,0.8884,0.8570,0.7448,0.7833,0.8828,0.8884,0.8781,0.8981,0.9734,0.9342,7405.0,0.8159,0.5163,0.6324,1691.0,0.7448,0.8998,0.7442,0.8676,0.7961,0.7290,0.7547,0.8577,0.8676,0.8592,0.8942,0.9498,0.9211,1851.0,0.6981,0.5083,0.5882,423.0,0.7290,0.8826,0.6684
2,P13_TFIDF_SelectKBest_LogReg,0.8883,0.9278,0.7037,0.7563,0.8984,0.8883,0.8690,0.8810,0.9976,0.9357,7405.0,0.9747,0.4098,0.5770,1691.0,0.7037,0.9447,0.8668,0.8663,0.8792,0.6553,0.6969,0.8699,0.8663,0.8392,0.8643,0.9914,0.9235,1851.0,0.8940,0.3191,0.4704,423.0,0.6553,0.9076,0.7493
3,P14_TFIDF_SelectKBest_LinearSVC,0.9594,0.9666,0.8982,0.9278,0.9601,0.9594,0.9578,0.9563,0.9957,0.9756,7405.0,0.9769,0.8007,0.8801,1691.0,0.8982,0.9868,0.9639,0.8949,0.8664,0.7613,0.7989,0.8901,0.8949,0.8862,0.9042,0.9741,0.9378,1851.0,0.8286,0.5485,0.6600,423.0,0.7613,0.9121,0.7776
4,P15_TFIDF_ComplementNB,0.9454,0.9101,0.9092,0.9096,0.9453,0.9454,0.9453,0.9661,0.9668,0.9665,7405.0,0.8541,0.8516,0.8528,1691.0,0.9092,0.9740,0.9289,0.8918,0.8310,0.7959,0.8116,0.8874,0.8918,0.8888,0.9208,0.9487,0.9345,1851.0,0.7411,0.6430,0.6886,423.0,0.7959,0.9011,0.7695


## Classement interne de chaque notebook

Les pipelines sont classés selon l’ordre de priorité suivant : `test_f1_class_1`, puis `test_recall_class_1`, puis `test_f1_macro`, puis `test_balanced_accuracy`, puis `test_roc_auc`. Les écarts train-test sont aussi calculés lorsqu’ils sont disponibles.


In [21]:
ranked_results = {}

for notebook_name, df in raw_results.items():
    print("=" * 100)
    print(f"Classement interne -> {notebook_name}")
    ranked_df = rank_pipelines(df)
    ranked_results[notebook_name] = ranked_df.copy()
    display(ranked_df)


Classement interne -> NB1


,rang,pipeline,train_f1_class_1,test_f1_class_1,train_recall_class_1,test_recall_class_1,train_precision_class_1,test_precision_class_1,train_f1_macro,test_f1_macro,train_balanced_accuracy,test_balanced_accuracy,train_roc_auc,test_roc_auc,train_pr_auc,test_pr_auc,gap_f1_class_1,gap_f1_macro
0,1,P05_TFIDF_UniBi_SGDLog,0.9899,0.6944,0.9846,0.6123,0.9952,0.8019,0.9938,0.8172,0.9918,0.7889,0.9999,0.9203,0.9994,0.7843,0.2955,0.1766
1,2,P01_Count_MultinomialNB,0.8155,0.6901,0.8427,0.7187,0.7899,0.6638,0.8858,0.8078,0.8958,0.8177,0.9610,0.9054,0.8981,0.7616,0.1254,0.0780
2,3,P04_TFIDF_UniBi_LinearSVC,0.9881,0.6869,0.9805,0.6147,0.9958,0.7784,0.9927,0.8122,0.9898,0.7873,0.9999,0.9161,0.9993,0.7834,0.3012,0.1805
3,4,P02_TFIDF_Unigram_LogReg,0.6583,0.5378,0.4991,0.3948,0.9668,0.8434,0.8011,0.7324,0.7476,0.6890,0.9638,0.9133,0.8970,0.7502,0.1205,0.0687
4,5,P03_TFIDF_UniBi_LogReg,0.6288,0.4949,0.4619,0.3428,0.9849,0.8896,0.7851,0.7101,0.7301,0.6665,0.9768,0.9157,0.9313,0.7677,0.1339,0.0750


Classement interne -> NB2


,rang,pipeline,train_f1_class_1,test_f1_class_1,train_recall_class_1,test_recall_class_1,train_precision_class_1,test_precision_class_1,train_f1_macro,test_f1_macro,train_balanced_accuracy,test_balanced_accuracy,train_roc_auc,test_roc_auc,train_pr_auc,test_pr_auc,gap_f1_class_1,gap_f1_macro
0,1,P10_WordChar_Hybrid_LinearSVC,0.9970,0.6797,0.9959,0.6170,0.9982,0.7565,0.9982,0.8073,0.9977,0.7858,1.0000,0.9200,1.0000,0.7757,0.3173,0.1909
1,2,P08_CharTFIDF_LinearSVC,0.9778,0.6471,0.9616,0.5745,0.9945,0.7409,0.9864,0.7887,0.9802,0.7643,0.9997,0.9141,0.9986,0.7524,0.3307,0.1977
2,3,P09_WordChar_Hybrid_LogReg,0.8075,0.5988,0.6836,0.4657,0.9863,0.8383,0.8858,0.7655,0.8407,0.7226,0.9899,0.9233,0.9658,0.7809,0.2087,0.1203
3,4,P07_CharTFIDF_LogReg,0.6739,0.5172,0.5169,0.3735,0.9679,0.8404,0.8098,0.7211,0.7565,0.6787,0.9709,0.9087,0.9111,0.7455,0.1567,0.0887
4,5,P06_TFIDF_Sublinear_LogReg,0.6302,0.4983,0.4636,0.3452,0.9837,0.8957,0.7858,0.7120,0.7309,0.6680,0.9773,0.9165,0.9324,0.7687,0.1319,0.0738


Classement interne -> NB3


,rang,pipeline,train_f1_class_1,test_f1_class_1,train_recall_class_1,test_recall_class_1,train_precision_class_1,test_precision_class_1,train_f1_macro,test_f1_macro,train_balanced_accuracy,test_balanced_accuracy,train_roc_auc,test_roc_auc,train_pr_auc,test_pr_auc,gap_f1_class_1,gap_f1_macro
0,1,P15_TFIDF_ComplementNB,0.8528,0.6886,0.8516,0.6430,0.8541,0.7411,0.9096,0.8116,0.9092,0.7959,0.9740,0.9011,0.9289,0.7695,0.1642,0.0980
1,2,P14_TFIDF_SelectKBest_LinearSVC,0.8801,0.6600,0.8007,0.5485,0.9769,0.8286,0.9278,0.7989,0.8982,0.7613,0.9868,0.9121,0.9639,0.7776,0.2201,0.1289
2,3,P12_TFIDF_SVD_LinearSVC,0.6324,0.5882,0.5163,0.5083,0.8159,0.6981,0.7833,0.7547,0.7448,0.7290,0.8998,0.8826,0.7442,0.6684,0.0442,0.0286
3,4,P11_TFIDF_SVD_LogReg,0.5973,0.5718,0.4666,0.4704,0.8297,0.7289,0.7644,0.7472,0.7224,0.7152,0.8972,0.8852,0.7405,0.6726,0.0255,0.0172
4,5,P13_TFIDF_SelectKBest_LogReg,0.5770,0.4704,0.4098,0.3191,0.9747,0.8940,0.7563,0.6969,0.7037,0.6553,0.9447,0.9076,0.8668,0.7493,0.1066,0.0594


## Sélection des candidats retenus

Pour le cœur du projet, on garde ici les **2 meilleurs pipelines** de chaque notebook ML.


In [22]:
selected_candidates = {}

for notebook_name, ranked_df in ranked_results.items():
    k = TOP_K[notebook_name]
    selected_df = ranked_df.head(k).copy()
    selected_candidates[notebook_name] = selected_df

    print("=" * 100)
    print(f"Candidats retenus -> {notebook_name} (top {k})")
    display(selected_df)


Candidats retenus -> NB1 (top 2)


,rang,pipeline,train_f1_class_1,test_f1_class_1,train_recall_class_1,test_recall_class_1,train_precision_class_1,test_precision_class_1,train_f1_macro,test_f1_macro,train_balanced_accuracy,test_balanced_accuracy,train_roc_auc,test_roc_auc,train_pr_auc,test_pr_auc,gap_f1_class_1,gap_f1_macro
0,1,P05_TFIDF_UniBi_SGDLog,0.9899,0.6944,0.9846,0.6123,0.9952,0.8019,0.9938,0.8172,0.9918,0.7889,0.9999,0.9203,0.9994,0.7843,0.2955,0.1766
1,2,P01_Count_MultinomialNB,0.8155,0.6901,0.8427,0.7187,0.7899,0.6638,0.8858,0.8078,0.8958,0.8177,0.9610,0.9054,0.8981,0.7616,0.1254,0.0780


Candidats retenus -> NB2 (top 2)


,rang,pipeline,train_f1_class_1,test_f1_class_1,train_recall_class_1,test_recall_class_1,train_precision_class_1,test_precision_class_1,train_f1_macro,test_f1_macro,train_balanced_accuracy,test_balanced_accuracy,train_roc_auc,test_roc_auc,train_pr_auc,test_pr_auc,gap_f1_class_1,gap_f1_macro
0,1,P10_WordChar_Hybrid_LinearSVC,0.9970,0.6797,0.9959,0.6170,0.9982,0.7565,0.9982,0.8073,0.9977,0.7858,1.0000,0.9200,1.0000,0.7757,0.3173,0.1909
1,2,P08_CharTFIDF_LinearSVC,0.9778,0.6471,0.9616,0.5745,0.9945,0.7409,0.9864,0.7887,0.9802,0.7643,0.9997,0.9141,0.9986,0.7524,0.3307,0.1977


Candidats retenus -> NB3 (top 2)


,rang,pipeline,train_f1_class_1,test_f1_class_1,train_recall_class_1,test_recall_class_1,train_precision_class_1,test_precision_class_1,train_f1_macro,test_f1_macro,train_balanced_accuracy,test_balanced_accuracy,train_roc_auc,test_roc_auc,train_pr_auc,test_pr_auc,gap_f1_class_1,gap_f1_macro
0,1,P15_TFIDF_ComplementNB,0.8528,0.6886,0.8516,0.6430,0.8541,0.7411,0.9096,0.8116,0.9092,0.7959,0.9740,0.9011,0.9289,0.7695,0.1642,0.0980
1,2,P14_TFIDF_SelectKBest_LinearSVC,0.8801,0.6600,0.8007,0.5485,0.9769,0.8286,0.9278,0.7989,0.8982,0.7613,0.9868,0.9121,0.9639,0.7776,0.2201,0.1289


## Export des sélections

Chaque sélection est exportée séparément dans `outputs/NB8/`. Aucun tableau concaténé n’est produit ici.


In [23]:
EXPORT_DIR = OUTPUTS_ROOT / "NB8"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for notebook_name, selected_df in selected_candidates.items():
    export_path = EXPORT_DIR / f"{notebook_name}_candidats_retenus.csv"
    selected_df.to_csv(export_path, index=False)
    print(f"Exporté : {export_path}")


Exporté : ..\..\outputs\NB8\NB1_candidats_retenus.csv
Exporté : ..\..\outputs\NB8\NB2_candidats_retenus.csv
Exporté : ..\..\outputs\NB8\NB3_candidats_retenus.csv


## Résumé final

À l’issue de ce notebook, tu auras une lecture claire des résultats de `NB1` à `NB4`, un classement interne pour chacun, puis une présélection des candidats officiellement retenus pour l’étape d’optimisation.
